# 03 — Two-Stage Default · REG · et

Stage 2 회귀 (y>0 only conditional, `E[Y|Y>0,x]`) → die-level reg_pred csv → combine 단계에서 clf prob과 곱.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/reg/et/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` (strategy_common §1)
- **TARGET_TRANSFORM**: `'none'` 고정 (strategy_common §24 — log1p_check 검증)
- **HPO**: N_TRIALS=100, **anchor enqueue + Narrow ±30%**
  - 1차 anchor는 log1p ON 컨텍스트지만 시작점으로 활용 — Optuna가 OFF 환경에 맞게 재탐색
- **Sampler/Pruner/Timeout**: §4·§25


## 1. 환경 설정 + import

In [ ]:
import os, sys

GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'
GDRIVE_MODELING_ID     = ''  # ★ Colab 사용 시 신규 modeling.zip ID 입력

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo, models

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')


## 2. 실험 설정

In [ ]:
# ── 모델 ──
REG_MODEL_NAME = 'et'
assert REG_MODEL_NAME in models.AVAILABLE_MODELS

# ── 실험 식별 ──
EXP_ID   = f'ts-reg-{REG_MODEL_NAME}-002'
EXP_MEMO = f'Two-Stage default · REG · {REG_MODEL_NAME} · y>0 conditional · transform=none'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS         = 100
N_FOLDS          = 5
N_STARTUP_TRIALS = 10
N_JOBS           = 7   # ★ strategy_common §8
TIMEOUT_SEC      = None  # ★ §25

# ── Two-Stage Stage 2 정책 ──
TARGET_TRANSFORM = 'none'   # ★ strategy_common §24 (log1p_check 검증: none=log1p 동등)
Y_POSITIVE_ONLY  = True
CLIP_Y_EXTREME   = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'reg', REG_MODEL_NAME)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 트리 PP_FIXED ──
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# ── 1차 best anchor (strategy.md §4.3, log1p ON 컨텍스트) — log1p OFF에서 재탐색 시작점 ──
REG_ANCHOR = {'n_estimators': 753, 'max_depth': 10, 'min_samples_leaf': 37, 'min_samples_split': 31, 'max_features': 'sqrt'}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'REG_MODEL_NAME: {REG_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | Y_POSITIVE_ONLY={Y_POSITIVE_ONLY}')
print(f'OUT_DIR={OUT_DIR}')


## 3. 데이터 로드 + Y clip + 전처리 (target_transform=none)

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# target transform: 'none' 고정 (strategy_common §24)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM} (strategy_common §24 — 트리 target_transform=none 통일)')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

yt = ys_input['train'][TARGET_COL]
print(f'\nUnit y>0 비율 (train): {(yt > 0).mean():.4f} ({(yt > 0).sum():,} unit)')
print(f'E[Y | Y>0]            = {yt[yt > 0].mean():.6f}')


## 4. Optuna HPO (REG, y>0 die만 학습)

- `run_hpo(y_positive_only=True, target_transform_fn=None, target_inverse_fn=None)`
- **Sampler/Pruner/Timeout**: §4·§25
- **anchor enqueue**: 1차 best HP (log1p ON 컨텍스트) — 시작점, Optuna가 재탐색
- 손실함수는 anchor에서 빼고 Optuna가 재선택 (log1p OFF 환경)

In [ ]:
study_meta = {
    'exp_id':           EXP_ID,
    'exp_memo':         EXP_MEMO,
    'user':             USER,
    'reg_model_name':   REG_MODEL_NAME,
    'target_transform': TARGET_TRANSFORM,
    'y_positive_only':  Y_POSITIVE_ONLY,
    'clip_y_extreme':   CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':         N_TRIALS,
    'n_folds':          N_FOLDS,
    'n_jobs':           N_JOBS,
    'timeout_sec':      TIMEOUT_SEC,
    'seed':             SEED,
    'anchor':           REG_ANCHOR,
    'sampler':          'TPE seed=None multivariate group',
    'pruner':           f'MedianPruner n_warmup=10',
}

sampler = TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS)
pruner  = MedianPruner(n_warmup_steps=10)

res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    user_attrs=study_meta,
    sampler=sampler,
    pruner=pruner,
    enqueue_trials=[REG_ANCHOR],   # ★ anchor 첫 trial 강제 (§5)
    timeout=TIMEOUT_SEC,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']

print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'best_params = {best_params}')


## 5. Best trial 재학습 (K-fold OOF) + die-level reg_pred 캐쳐

In [ ]:
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=REG_MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    y_positive_only=Y_POSITIVE_ONLY,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

y_train_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_true   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_true  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

oof_unit  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_train_true.index]
val_unit  = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
test_unit = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]

oof_rmse  = float(np.sqrt(np.mean((oof_unit.values  - y_train_true.values) ** 2)))
val_rmse  = float(np.sqrt(np.mean((val_unit.values  - y_val_true.values)   ** 2)))
test_rmse = float(np.sqrt(np.mean((test_unit.values - y_test_true.values)  ** 2)))

print(f'\n[Refit 완료] (reg 단독, y>0 conditional, transform=none)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')


## 6. 산출물 저장

In [ ]:
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=None,   # ★ combine 단계에서 후처리 (여기선 단순 mean)
    study_meta=study_meta,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'reg_{REG_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass
